In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-05 18:27:22--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-06T03%3A15%3A43Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-06T02%3A14%3A46Z&ske=2026-03-06T03%3A15%3A43Z&sks=b&skv=2018-11-09&sig=iEMZeEdCH%2F7d%2Fmc7Bx3DRCPhnu8MdIDdlWVF5gUlnHk%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Mjc2NzY0NCwibmJmIjoxNzcyNzY0MDQ0LCJwYXRo

In [4]:

!ls fhvhv_tripdata_2021-01.csv


fhvhv_tripdata_2021-01.csv


In [5]:
!ls -ltr fhvhv_tripdata_2021-01.csv

-rw-r--r-- 1 annap 197609 752335705 Jul 14  2022 fhvhv_tripdata_2021-01.csv


In [4]:
!gzip -d fhvhv_tripdata_2021-01.csv.gz

gzip: fhvhv_tripdata_2021-01.csv already exists;	not overwritten


In [6]:
!wc -l fhvhv_tripdata_2021-01.csv

11908469 fhvhv_tripdata_2021-01.csv


In [7]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [8]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [9]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [11]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd

In [13]:
df_pandas = pd.read_csv('head.csv')

In [11]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [17]:
df_pandas.head()

,hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
0,HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,NaN
1,HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,NaN
2,HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,NaN
3,HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,NaN
4,HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,NaN


In [14]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [13]:
from pyspark.sql import types

In [14]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [15]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [16]:
df = df.repartition(24)

In [17]:
df.write.parquet('fhvhv/2021/01/')

In [18]:
df = spark.read.parquet('fhvhv/2021/01/')

In [19]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [21]:
from pyspark.sql import functions as F

In [22]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-01 18:53:52|2021-01-01 19:21:31|          78|         265|   NULL|
|           HV0003|              B02879|2021-01-01 05:25:47|2021-01-01 05:52:53|         100|          37|   NULL|
|           HV0003|              B02865|2021-01-03 14:16:13|2021-01-03 14:38:09|         177|          76|   NULL|
|           HV0005|              B02510|2021-01-03 00:12:19|2021-01-03 00:28:03|         145|          90|   NULL|
|           HV0003|              B02876|2021-01-01 11:08:59|2021-01-01 11:27:42|         243|         186|   NULL|
|           HV0003|              B02876|2021-01-01 00:59:31|2021-01-01 01:28:44|

In [24]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003').head(5)

[Row(pickup_datetime=datetime.datetime(2021, 1, 1, 5, 25, 47), dropoff_datetime=datetime.datetime(2021, 1, 1, 5, 52, 53), PULocationID=100, DOLocationID=37),
 Row(pickup_datetime=datetime.datetime(2021, 1, 3, 14, 16, 13), dropoff_datetime=datetime.datetime(2021, 1, 3, 14, 38, 9), PULocationID=177, DOLocationID=76),
 Row(pickup_datetime=datetime.datetime(2021, 1, 1, 11, 8, 59), dropoff_datetime=datetime.datetime(2021, 1, 1, 11, 27, 42), PULocationID=243, DOLocationID=186),
 Row(pickup_datetime=datetime.datetime(2021, 1, 1, 0, 59, 31), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 28, 44), PULocationID=116, DOLocationID=49),
 Row(pickup_datetime=datetime.datetime(2021, 1, 3, 3, 18, 1), dropoff_datetime=datetime.datetime(2021, 1, 3, 3, 38, 7), PULocationID=16, DOLocationID=53)]

In [18]:
spark.stop()